# Annular Boussinesq — AWFNO JCP one-run numerical study

Google Colab GPU: select **Runtime → Run all**. This executes two suites sequentially and stores checkpoints and results on Google Drive so completed trials can be resumed.

1. **Original sample-size sweep:** TB-AWFNO, AWFNO, WFNO, full-trajectory FNO at N = 25, 50, 100, 200, 400.
2. **Controlled reviewer suites (three seeds):** LGL / LLG / GLL orderings, LLL / GGG component ablations, window K = 1, 10, 30, radius r = 1, 2, 3, Fourier modes (8,12)/(16,24)/(24,32), parameter-matched WFNO and temporal-basis head.

The observable is the scalar HyTeG annulus field `c_grid`, not velocity or pressure. This notebook does **not** implement modern F-FNO, U-Net, CNO, or U-NO baselines. The controlled study has 51 fits plus up to 20 original fits, so it may require more than one Colab session. Re-executing skips saved completed trials.

In [ ]:
#@title 1. Mount Drive and check annulus data
from google.colab import drive
from pathlib import Path
import torch, os, sys, subprocess
drive.mount('/content/gdrive')
DATASET=Path('/content/gdrive/MyDrive/anus/hyteg_annulus_c_grid_64x128_samples500.pt')
OUTPUT_ROOT=Path('/content/gdrive/MyDrive/anus/Boussinesq_JCP')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
assert DATASET.is_file(), f"Missing dataset: {DATASET}"
assert torch.cuda.is_available(), "Set Google Colab runtime to GPU."
print('GPU:',torch.cuda.get_device_name(0))
print('Data:',DATASET,'size GiB:',round(DATASET.stat().st_size/1024**3,3))
print('Outputs:',OUTPUT_ROOT)

In [ ]:
#@title 2. Install the reference full-trajectory FNO dependency
subprocess.check_call([sys.executable,'-m','pip','-q','install','neuraloperator==2.0.0'])
import importlib.metadata
print('NeuralOperator:',importlib.metadata.version('neuraloperator'))

## Complete experiment engine

The driver is embedded in the following cell with SHA256 integrity verification. It generates `/content/annulus2d_jcp_experiments.py`; no second upload is needed.

In [ ]:
import base64, gzip, hashlib
from pathlib import Path
B64_DRIVER = """..."""
DRIVER = Path('/content/annulus2d_jcp_experiments.py')
source = gzip.decompress(base64.b64decode(B64_DRIVER))
assert hashlib.sha256(source).hexdigest() == '80266deab9841c253acf610507015ad5294923d7e72f7a4894759e9da36aa86a', 'Embedded driver checksum mismatch'
DRIVER.write_bytes(source)
print('Wrote', DRIVER, '|', len(source), 'bytes')


In [ ]:
#@title 4. Configuration — full original + controlled runs
RUN_ORIGINAL = True #@param {type:'boolean'}
RUN_CONTROLLED = True #@param {type:'boolean'}
CONTROLLED_SUITE = 'all' #@param ['all','ordering','components','window','radius','modes','matched','temporal_basis']
SEEDS = '1234,1235,1236' #@param {type:'string'}
SAMPLE_SIZES = '25,50,100,200,400' #@param {type:'string'}
N_TRAIN_CONTROLLED = 400 #@param {type:'integer'}
EPOCHS = 150 #@param {type:'integer'}
BATCH_SIZE = 32 #@param {type:'integer'}
FORCE_RETRAIN = False #@param {type:'boolean'}
WINDOW_SWEEP = '1,10,30'
RADIUS_SWEEP = '1,2,3'
assert RUN_ORIGINAL or RUN_CONTROLLED
print('Original:',RUN_ORIGINAL,'Controlled:',RUN_CONTROLLED,'Seeds:',SEEDS)

In [ ]:
#@title 5. Run each selected suite (Drive checkpoints allow resumption)
cfg=os.environ.copy()
cfg.update({
  'DATA_PATH':str(DATASET),'OUTPUT_ROOT':str(OUTPUT_ROOT),
  'RUN_SAMPLE_SIZES':SAMPLE_SIZES,
  'EPOCHS':str(EPOCHS),'BATCH_SIZE':str(BATCH_SIZE),
  'SPLIT_MODE':'contiguous','SPLIT_SEED':'2026',
  'N_VAL':'30','N_TEST':'70',
  'WINDOW_K':'30','WINDOW_STRIDE':'30','N_TIME_MODES':'18',
  'WIDTH':'32','N_LAYERS':'4','MODES_R':'16','MODES_THETA':'24',
  'LOCAL_RADIUS':'2',
  'REBUTTAL_N_TRAIN':str(N_TRAIN_CONTROLLED),
  'REBUTTAL_SEEDS':SEEDS,'REBUTTAL_RUN':CONTROLLED_SUITE,
  'REBUTTAL_OUTPUT':str(OUTPUT_ROOT/'jcp_controlled_annulus'),
  'WINDOW_SWEEP':WINDOW_SWEEP,'RADIUS_SWEEP':RADIUS_SWEEP,
  'FORCE_RETRAIN':'1' if FORCE_RETRAIN else '0','PYTHONUNBUFFERED':'1'
})
for stage,enabled in [('original',RUN_ORIGINAL),('rebuttal',RUN_CONTROLLED)]:
    if not enabled:continue
    env=dict(cfg);env['EXPERIMENT_SUITE']=stage
    print('START:',stage,flush=True)
    subprocess.run([sys.executable,'-u',str(DRIVER)],check=True,env=env)
    print('DONE:',stage,flush=True)
print('Finished selected experiments:',OUTPUT_ROOT)

## Saved summaries

The final cell is safe to rerun independently after interrupted training. It displays completed summaries only and calculates mean/standard deviation across available controlled seeds (not across sample sizes).

In [ ]:
#@title 6. Aggregate available experimental results
import json
import pandas as pd
from IPython.display import display
for source in sorted(OUTPUT_ROOT.rglob('sample_size_sweep_summary_run_*.json')):
    print('Original sample-size results:',source)
    display(pd.DataFrame(json.loads(source.read_text())))
p=OUTPUT_ROOT/'jcp_controlled_annulus'/'results.csv'
if p.exists():
    raw=pd.read_csv(p)
    print('Controlled trials:',len(raw))
    display(raw.head(10))
    metrics=[x for x in ['full_l2_mean','final_l2_mean','low_theta_l2','high_theta_l2','sharp_region_l2','train_seconds','rollout_seconds','numel_parameters','real_scalar_dof','peak_rollout_gpu_MiB'] if x in raw.columns]
    agg=raw.groupby('experiment',as_index=False)[metrics].agg(['mean','std'])
    agg.columns=['_'.join(filter(None,map(str,col))) for col in agg.columns.to_flat_index()]
    out=p.parent/'results_aggregated_by_seed.csv'
    agg.to_csv(out,index=False)
    print('Aggregated:',out)
    display(agg)
else:
    print('Controlled trials not started:',p)